To make this agent truly "Smart," we don't want the LLM to read the entire dictionary.json every single time (which is slow and expensive). Instead, we will convert the Semantic Context into mathematical vectors and store them in ChromaDB. This uses the ***sentence-transformers***. This is implementing ***RAG (Retrieval-Augmented Generation)***.

**When a user asks:** "Why is the Idaho route failing?", the agent will perform a Similarity Search in ChromaDB to grab only the relevant "Geospatial Risk" logic.

Since, the first making of "dictionary" missed some of the KPIs, and Red Flags. So, after updating it, we need to make sure that these are re-indexed in ChromaDB.

In [6]:
import json
import os
import chromadb
from chromadb.utils import embedding_functions

# 1. Path Configuration
PROJECT_ROOT = r"B:\3. Prog\2. Projects\7. Logistics and supply chain"
DICT_PATH = os.path.join(PROJECT_ROOT, "metadata", "dictionary.json")
CHROMA_PATH = os.path.join(PROJECT_ROOT, "chroma_db")

# 2. Load the Validated Dictionary (40 items)
with open(DICT_PATH, 'r', encoding='utf-8') as f:
    intelligence_data = json.load(f)

# 3. Initialize ChromaDB Persistent Client
client = chromadb.PersistentClient(path=CHROMA_PATH)

# Using the standard lightweight embedding model
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# 4. Create/Refresh Collection
COLLECTION_NAME = "logistics_logic"
try:
    client.delete_collection(COLLECTION_NAME)
    print(f"🧹 Old collection '{COLLECTION_NAME}' purged.")
except:
    pass

collection = client.create_collection(name=COLLECTION_NAME, embedding_function=emb_fn)

# 5. Indexing the Intelligence
print(f"🚀 Indexing {len(intelligence_data)} items into Vector DB...")

for i, item in enumerate(intelligence_data):
    # We combine Name and Description for the search document
    searchable_text = f"{item['name']}: {item['description']}"
    
    collection.add(
        ids=[f"id_{i:02d}"],
        documents=[searchable_text],
        metadatas=[{
            "name": item['name'],
            "sql_snippet": item.get('sql', ""),
            "target_tables": ", ".join(item.get('tables', []))
        }]
    )

# 6. Final Verification
total_indexed = collection.count()
print("-" * 40)
print(f"✅ SUCCESS: ChromaDB is live at {CHROMA_PATH}")
print(f"✅ Total Semantic Vectors Indexed: {total_indexed}")
print("-" * 40)

🧹 Old collection 'logistics_logic' purged.
🚀 Indexing 40 items into Vector DB...
----------------------------------------
✅ SUCCESS: ChromaDB is live at B:\3. Prog\2. Projects\7. Logistics and supply chain\chroma_db
✅ Total Semantic Vectors Indexed: 40
----------------------------------------
